# 🏟️ Stadium Vision System — Backend en Google Colab

Levanta el backend Flask + YOLOv8n y lo expone por HTTPS para que el frontend
desplegado en Vercel pueda consumirlo.

**Antes de empezar:** activa la GPU en `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.
Sin GPU funciona igual, pero el stream irá bastante más lento.

Ejecuta las celdas **en orden**. La última imprime la URL que debes pegar en el frontend.

## 1. Clonar el repositorio

In [ ]:
import os, shutil

REPO = 'https://github.com/miguelbello2/primerParcialSw2.git'
DEST = '/content/primerParcialSw2'

# Empezar siempre desde un clon limpio evita arrastrar estado de sesiones previas
if os.path.exists(DEST):
    shutil.rmtree(DEST)

!git clone --depth 1 $REPO $DEST
%cd $DEST/backend
!ls -la

## 2. Instalar dependencias

Colab ya trae `torch`, `numpy` y `opencv`, así que solo instalamos lo que falta.
Tarda ~1-2 minutos.

In [ ]:
!pip install -q flask==2.3.3 flask-cors==4.0.0 flask-limiter==3.5.0 werkzeug==2.3.7 gunicorn==21.2.0 ultralytics==8.3.50

import torch
print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())

## 3. Preparar los pesos de YOLOv8n

Los pesos vienen versionados en `models/`. Esta celda los copia a donde el
backend los busca (su directorio de trabajo) y verifica que carguen.

In [ ]:
import shutil, os
from pathlib import Path

os.makedirs('models', exist_ok=True)
src = Path('../models/yolov8n.pt')
dst = Path('models/yolov8n.pt')

if src.exists() and not dst.exists():
    shutil.copy(src, dst)
    print(f'Pesos copiados: {dst} ({dst.stat().st_size / 1e6:.1f} MB)')
elif dst.exists():
    print(f'Pesos ya presentes: {dst} ({dst.stat().st_size / 1e6:.1f} MB)')
else:
    print('Sin pesos locales; ultralytics los descargará al primer uso.')

# Carga anticipada: mejor fallar aquí que en el primer request del frontend
from ultralytics import YOLO
_m = YOLO(str(dst) if dst.exists() else 'yolov8n.pt')
print('Modelo cargado OK')

## 4. Arrancar el backend

Usamos **gunicorn con 1 worker y muchos hilos**: el estado del backend
(`active_tasks`, `stream_stats`, `latest_uploaded_file`) vive en memoria del
proceso, así que con más de un worker cada request caería en un proceso
distinto y el dashboard leería datos vacíos.

`--timeout 0` es obligatorio: el endpoint MJPEG mantiene la respuesta abierta
indefinidamente y gunicorn mataría al worker con el timeout por defecto.

In [ ]:
import subprocess, time, requests, os, signal

PORT = 5000

# Matar cualquier backend de una ejecución anterior de esta misma celda
!pkill -f gunicorn 2>/dev/null; sleep 1

log = open('/content/backend.log', 'w')
server = subprocess.Popen(
    ['gunicorn', '--bind', f'0.0.0.0:{PORT}',
     '--workers', '1', '--threads', '16',
     '--timeout', '0', '--log-level', 'info',
     'app:app'],
    stdout=log, stderr=subprocess.STDOUT,
)

# Esperar a que /health responda antes de seguir
ready = False
for _ in range(40):
    time.sleep(1)
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/health', timeout=2).ok:
            ready = True
            break
    except requests.RequestException:
        pass

if ready:
    print(f'✅ Backend escuchando en el puerto {PORT}')
else:
    print('❌ El backend no respondió. Log:')
    print(open('/content/backend.log').read()[-3000:])

## 5. Exponer con Cloudflare Tunnel

Preferimos `cloudflared` sobre ngrok por dos razones: no necesita cuenta ni
token, y no interpone la página de advertencia que ngrok muestra en su plan
gratuito (esa advertencia es lo que rompe el `<img>` del stream MJPEG).

La URL es distinta en cada sesión — es normal, el frontend permite cambiarla
sin redeployar.

In [ ]:
import subprocess, time, re, os

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

!pkill -f cloudflared 2>/dev/null; sleep 1

tunnel_log = '/content/cloudflared.log'
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}',
     '--no-autoupdate', '--logfile', tunnel_log],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

public_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists(tunnel_log):
        match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open(tunnel_log).read())
        if match:
            public_url = match.group(0)
            break

if public_url:
    print('=' * 70)
    print('  URL PÚBLICA DEL BACKEND:')
    print(f'  {public_url}')
    print('=' * 70)
else:
    print('❌ No se obtuvo la URL. Log:')
    print(open(tunnel_log).read()[-2000:] if os.path.exists(tunnel_log) else 'sin log')

## 6. Verificar de extremo a extremo

In [ ]:
import requests

for path in ['/health', '/api/results/latest', '/api/stats/overview']:
    try:
        r = requests.get(public_url + path, timeout=20)
        print(f'{r.status_code}  {path}  →  {r.text[:120]}')
    except Exception as e:
        print(f'ERR  {path}  →  {e}')

FRONTEND = 'https://TU-PROYECTO.vercel.app'  # ← reemplaza por tu dominio de Vercel
print(f'\n👉 Abre el frontend ya apuntando al backend:\n{FRONTEND}/?api={public_url}')

## 7. Mantener viva la sesión

Colab desconecta el entorno tras ~90 minutos sin interacción y corta cualquier
sesión a las 12 horas. Ejecuta esta celda y **déjala corriendo**: mientras siga
activa, el backend sigue en pie y verás el conteo de detecciones en vivo.

Para detener todo: `Entorno de ejecución → Interrumpir ejecución`.

In [ ]:
import time, requests
from datetime import datetime

print(f'Backend vivo en {public_url} — mantén esta celda ejecutándose.\n')
try:
    while True:
        try:
            data = requests.get(f'http://127.0.0.1:{PORT}/api/results/latest', timeout=5).json()
            print(f"[{datetime.now():%H:%M:%S}] archivo={data.get('active_file_id')} "
                  f"densidad={data.get('density')} ánimo={data.get('mood')}")
        except Exception as e:
            print(f'[{datetime.now():%H:%M:%S}] sin respuesta: {e}')
        time.sleep(60)
except KeyboardInterrupt:
    print('\nDetenido por el usuario.')

---
### Diagnóstico

Si algo falla, esta celda muestra el log del backend.

In [ ]:
print(open('/content/backend.log').read()[-5000:])